# Практика 1. NumPy, pandas и форматы данных

Массивы и таблицы: выбор строк, пропуски, чтение и запись CSV, JSON и Parquet.

Нужны переменные, списки и функции Python. Для локальной работы установите библиотеки: `python -m pip install jupyter numpy pandas pyarrow`, затем запустите `jupyter notebook`. В облачном окружении проверьте доступность этих библиотек.

Выполняйте ячейки сверху вниз (`Shift + Enter`), в том числе после перезапуска ядра. В заданиях оставлены места для кода, после условий приведены ответы для проверки.

Файлы создаются в текущей рабочей папке. Используйте отдельную папку для практики, чтобы не перезаписать свои файлы.


In [ ]:
import numpy as np
import pandas as pd

## 1. NumPy: массивы

In [ ]:
a = np.array([1, 2, 3, 4, 5])
print(a)
print("тип:", type(a))
print("размер:", a.shape)

Операции применяются сразу ко всему массиву, без циклов:

In [ ]:
print(a + 10)
print(a * 2)
print(a ** 2)

In [ ]:
print("сумма:   ", a.sum())
print("среднее: ", a.mean())
print("минимум: ", a.min())
print("максимум:", a.max())

Двумерный массив (матрица). В этом примере строки — наблюдения, столбцы — признаки. Индексы NumPy начинаются с нуля: `m[1, 2]` — элемент второй строки и третьего столбца.


In [ ]:
m = np.array([
    [1, 2, 3],
    [4, 5, 6],
    [7, 8, 9],
])
print(m)
print("форма:", m.shape)
print("первая строка:", m[0])
print("первый столбец:", m[:, 0])
print("элемент [1, 2]:", m[1, 2])

Фильтрация по условию:

In [ ]:
print(a[a > 2])

Способы создать массив:


In [ ]:
print(np.zeros(3))
print(np.ones(3))
print(np.arange(0, 10, 2))
print(np.linspace(0, 1, 5))

### Задание 1
Создайте массив целых чисел от 1 до 10 включительно, возведите числа в квадрат и найдите сумму.

**Самопроверка:** форма исходного массива `(10,)`, первый квадрат — `1`, последний — `100`, сумма — `385`. Помните: правая граница `arange` не включается.


In [ ]:
# ваш код

## 2. pandas: таблицы

- `Series` — одномерные данные с индексом.
- `DataFrame` — таблица с индексом строк и именами столбцов.

Индекс — метка строки, она не обязана совпадать с её позицией.


In [ ]:
s = pd.Series([10, 20, 30], name="числа")
s

In [ ]:
df = pd.DataFrame({
    "имя":     ["Аня", "Борис", "Вера", "Глеб", "Дина"],
    "возраст": [23, 31, 27, 45, 38],
    "город":   ["Москва", "Казань", "Москва", "Сочи", "Казань"],
    "зарплата": [90000, 120000, 105000, None, 98000],
})
df

### Осмотр таблицы

In [ ]:
print("форма:", df.shape)
df.head(3)

In [ ]:
df.info()

In [ ]:
df.describe()

### Столбцы и строки

In [ ]:
print(df["имя"])

In [ ]:
df[["имя", "возраст"]]

In [ ]:
print(df.loc[0])   # строка с меткой индекса 0
print(df.iloc[0])  # строка на позиции 0 (первая строка)

# Теперь метками строк служат имена; позиция первой строки не изменилась.
by_name = df.set_index("имя")
print(by_name.loc["Аня"])
print(by_name.iloc[0])


In [ ]:
df[df["возраст"] > 30]    # фильтр

In [ ]:
df[(df["город"] == "Москва") & (df["возраст"] > 25)]  # & соединяет условия; каждое в скобках


### Новые столбцы и сортировка

In [ ]:
df["возраст_через_5_лет"] = df["возраст"] + 5
df.sort_values("возраст", ascending=False)

### Группировка

In [ ]:
df.groupby("город")["возраст"].mean()

In [ ]:
df["город"].value_counts()

### Пропуски

`isna()` находит пропуски. При вычислении среднего pandas по умолчанию исключает их. Ниже заполняем пропуски средним; в другой задаче может потребоваться иной способ. Для машинного обучения среднее вычисляют только по обучающей выборке. Здесь используем одну учебную таблицу.


In [ ]:
print(df.isna().sum())

In [ ]:
# Сохраняем исходную таблицу, чтобы сравнить варианты обработки.
df_original = df.copy()
mean_salary = df["зарплата"].mean()
df["зарплата"] = df["зарплата"].fillna(mean_salary)
df


### Задание 2
1. Выберите людей строго старше 25 лет.
2. По таблице `df` после заполнения пропусков найдите среднюю зарплату по городам.
3. Добавьте столбец `зарплата_в_год` (зарплата × 12).

**Самопроверка:** выбраны 4 человека; средние по городам: Москва — `97500`, Казань — `109000`, Сочи — `103250`. Годовая зарплата Ани — `1080000`.


In [ ]:
# ваш код

## 3. Форматы данных

Одну таблицу можно записать разными способами:

| Формат | Особенности |
|---|---|
| **CSV** | Текстовая таблица; разделитель не обязательно запятая. Типы данных при чтении определяются заново. |
| **JSON** | Текстовые объекты и массивы. Представление таблицы зависит от выбранного `orient`. |
| **Parquet** | Бинарный формат с хранением по столбцам. Сохраняет типы и поддерживает сжатие. |

После чтения проверяйте не только размер таблицы, но и значения, типы столбцов и пропуски. Исходный индекс ниже не сохраняем (`index=False` для CSV и Parquet).


### CSV

In [ ]:
from pathlib import Path

df.to_csv("people.csv", index=False, encoding="utf-8")
print(Path("people.csv").read_text(encoding="utf-8"))


In [ ]:
df_csv = pd.read_csv("people.csv")
df_csv

### JSON

In [ ]:
df.to_json("people.json", orient="records", force_ascii=False, indent=2)
print(Path("people.json").read_text(encoding="utf-8"))


In [ ]:
df_json = pd.read_json("people.json")
df_json

Стандартный модуль `json` читает JSON-объект как словарь, JSON-массив как список. В примере ниже корневое значение — объект.


In [ ]:
import json

data = {"название": "Мой проект", "версия": 1, "теги": ["данные", "python"]}

with open("config.json", "w", encoding="utf-8") as f:
    json.dump(data, f, ensure_ascii=False, indent=2)

with open("config.json", encoding="utf-8") as f:
    loaded = json.load(f)

print(loaded)
print(loaded["теги"][0])

### Parquet

Для чтения и записи используем `pyarrow`, указанный среди зависимостей в начале. Parquet подходит для больших таблиц; на маленьком примере служебные данные могут сделать файл больше CSV.


In [ ]:
df.to_parquet("people.parquet", index=False, engine="pyarrow")
df_parquet = pd.read_parquet("people.parquet", engine="pyarrow")
df_parquet


### Из таблицы в numpy и обратно

In [ ]:
arr = df[["возраст", "зарплата"]].to_numpy()
print(arr)
print(arr.shape)

In [ ]:
pd.DataFrame(arr, columns=["возраст", "зарплата"])

Массивы numpy тоже можно сохранять:

In [ ]:
np.save("arr.npy", arr)
print(np.load("arr.npy"))

### Задание 3
1. Сохраните таблицу `df` в `result.csv` без индекса и прочитайте обратно.
2. Сравните форму, имена столбцов, типы данных и значения с исходной таблицей.
3. Повторите с `result.parquet`, затем сравните размеры файлов (`Path(...).stat().st_size`).

**Самопроверка:** форма совпадает с `df.shape`, а не с фиксированным числом: в задании 2 вы могли добавить столбец. Для проверки содержимого используйте `pd.testing.assert_frame_equal(df, loaded, check_dtype=False)`; типы сравните отдельно через `dtypes`. Успешная проверка ничего не выводит. Из-за текстового представления допустимы небольшие различия вещественных чисел. Не делайте вывод о размере форматов по одной маленькой таблице.


In [ ]:
# ваш код

## Вопросы

1. Почему `df.loc[0]` может не сработать после смены индекса? Чем отличается `iloc`?
2. Как изменится средняя зарплата по городам, если оставить пропуск?
3. Почему CSV и Parquet могут вернуть одинаковые значения с разными типами?
